# 第四节 基于Pytorch快速构建感知机实现手写数字二分类

## 实验目标
通过本案例的学习：

1. 掌握不依赖数学知识，使用深度学习框架快速实现模型结构定义、损失函数定义、梯度下降的方法；


## 注意事项

1. 本案例推荐使用Pytorch-1.0.0、CPU运行；

2. 如果您是第一次使用 JupyterLab，请查看[《ModelArts JupyterLab使用指导》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0012.html)了解使用方法；

3. 如果您在使用 JupyterLab 过程中碰到报错，请参考[《ModelArts JupyterLab常见问题解决办法》](https://support.huaweicloud.com/modelarts_faq/modelarts_05_0185.html)尝试解决问题。

## 实验步骤

### 案例内容介绍
上一节我们从零开始实现了感知机的模型结构、定义了损失函数和评价函数，并且手动推导了梯度下降的公式，最终经过3000个epoch的训练，使得感知机模型在手写数字0和1的二分类任务上达到了0.9以上的准确率。 
可能你已经感受到，这个从零开始实现整个机器学习过程的方式是比较费劲的，特别是在手动推导梯度下降公式这一块，需要一定的数学知识。如果我们更换另一个损失函数，那又得再重新推导一遍新的梯度下降公式，这对调模型来说是费力的事情。  
好在当今已经有很多的深度学习框架，它们已经友好地封装了模型结构定义、损失函数定义、梯度下降实现等过程，只需要进行一些简单的函数调用，就可以实现完成的机器学习训练过程，无需关注底层的梯度下降是如何实现的，极大地提高了模型开发的效率。  
下面，我们就用Pytorch框架来快速实现感知机模型，对手写数字进行二分类。

### 1. 加载数据集
由于上一节已经定义了load_data_zeros_ones函数，所以在本节我们直接进行调用即可

In [1]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '../datasets/MNIST_data'))
from load_data_zeros_ones import load_data_zeros_ones

datasets_dir = '../datasets'
train_x, train_y, test_x, test_y = load_data_zeros_ones(datasets_dir)

INFO:root:Using MoXing-v1.17.3-

INFO:root:Using OBS-Python-SDK-3.20.7


数字0，训练集规模： 5923 ，测试集规模： 980

数字1，训练集规模： 6742 ，测试集规模： 1135


load_data_zeros_ones函数返回的数据格式是np.ndarray格式，但是在Pytorch中要求的格式是torch.tensor格式，因此要执行下面的代码进行数据格式转换

In [2]:
import torch
import numpy as np

train_x = torch.tensor(train_x.astype(np.float32))
train_y = torch.tensor(train_y.astype(np.float32))
test_x = torch.tensor(test_x.astype(np.float32))
test_y = torch.tensor(test_y.astype(np.float32))

### 2. 定义网络结构
使用Pytorch实现感知机模型非常简单，只需要调用nn.Linear定义一个全连接层，再加上一个Sigmoid单元即可，并且nn.Linear会对权值w和阈值偏置b自动进行初始化，代码如下：

In [3]:
from torch import nn

class Network(nn.Module):
    def __init__(self, num_of_weights):
        torch.manual_seed(0)
        super().__init__()
        self.fc = nn.Linear(in_features=num_of_weights, out_features=1, bias=True)  # 定义一个全连接层
        self.nonlinearity = nn.Sigmoid()
    
    def forward(self, x):  # 加权求和单元和非线性函数单元通过定义计算过程来实现
        z = self.fc(x)
        pred_y = self.nonlinearity(z)
        return pred_y

### 3. 定义损失函数
Pytorch支持很多种损失函数，都定义torch.nn.functional模块中，我们直接使用该模块中的mse_loss函数，这就是均方误差函数，代码如下：

In [4]:
import torch.nn.functional as F
loss_fun = F.mse_loss

### 4. 定义评价函数
评价函数直接复用上一节的定义即可

In [5]:
class Network(nn.Module):
    def __init__(self, num_of_weights):
        torch.manual_seed(0)
        super().__init__()
        self.fc = nn.Linear(in_features=num_of_weights, out_features=1, bias=True)  # 定义一个全连接层
        self.nonlinearity = nn.Sigmoid()
    
    def forward(self, x):  # 加权求和单元和非线性函数单元通过定义计算过程来实现
        z = self.fc(x)
        pred_y = self.nonlinearity(z)
        return pred_y
   
    def evaluate(self, pred_y, true_y, threshold=0.5):
        pred_y[pred_y < threshold] = 0  # 预测值小于0.5，则判为类别0
        pred_y[pred_y >= threshold] = 1

        acc = (pred_y == true_y).float().mean()
        return acc

### 5. 一行代码实现梯度下降算法
Pytorch框架有一个特性，称为：自动微分，微分就是求导的意思，自动微分意味着Pytorch框架能对任意函数进行自动求导，也就是说使用Pytorch框架可以定义任意的网络、任意的损失函数，它都能自动地求导得出参数的梯度，根本就不需要我们进行任何的公式推导过程。  
Pytorch还支持很多种梯度下降的优化器，都定义在torch.optim模块中，我们只需要一行代码，直接调用该模块中的SGD函数即可，代码如下：

In [6]:
net = Network(28*28)  # 创建网络
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)  # 实现梯度下降

### 6. 实现训练函数
一个模型的训练过程就是：1）前向传播；2）计算损失；3）计算梯度；4）更新权值，把这些过程拼装起来即可

In [7]:
def train(net, train_x, train_y, test_x, test_y, max_epochs=100):
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(1, max_epochs + 1):
        net.train()  # 切换为训练模式
        pred_y_train = net.forward(train_x)  # 前向传播
        train_loss = loss_fun(pred_y_train, train_y)  # 计算损失

        # 计算梯度，更新权值
        train_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (epoch == 1) or (epoch % 10 == 0):
            net.eval()  # 切换为评价模式，评价模式不计算梯度，计算更快
            pred_y_test = net.forward(test_x)
            test_loss = loss_fun(pred_y_test, test_y)
            train_acc = net.evaluate(pred_y_train, train_y)
            test_acc = net.evaluate(pred_y_test, test_y)
            print('epoch %d, train_loss %.4f, test_loss %.4f, train_acc: %.4f, test_acc: %.4f' % (epoch, train_loss.item(), test_loss.item(), train_acc, test_acc))
    return train_losses, test_losses, train_accs, test_accs

### 7. 开始训练
训练耗时小于2秒

In [8]:
import time
start_time = time.time()
max_epochs = 50
train_losses, test_losses, train_accs, test_accs = train(net, train_x, train_y, test_x, test_y, max_epochs=max_epochs)
print('cost time: %.1f s' % (time.time() - start_time))

epoch 1, train_loss 0.2319, test_loss 0.2241, train_acc: 0.7596, test_acc: 0.8293

epoch 10, train_loss 0.1780, test_loss 0.1723, train_acc: 0.9811, test_acc: 0.9853

epoch 20, train_loss 0.1397, test_loss 0.1350, train_acc: 0.9914, test_acc: 0.9957

epoch 30, train_loss 0.1138, test_loss 0.1097, train_acc: 0.9928, test_acc: 0.9976

epoch 40, train_loss 0.0954, test_loss 0.0915, train_acc: 0.9935, test_acc: 0.9981

epoch 50, train_loss 0.0818, test_loss 0.0782, train_acc: 0.9938, test_acc: 0.9986

cost time: 1.4 s


从上面的结果可以看到，使用Pytorch实现的感知机模型，仅使用不到2秒的时间，训练了50个epoch之后就达到了0.9986的准确率，相比上一节的实现，训练又快又好，这说明使用Pytorch来进行模型的开发，不仅开发效率更高，实现的结果也更优，这就是使用深度学习框架带来的优势。

### 本节扩展学习材料
[torch.nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear)  
[优化器介绍](https://education.huaweicloud.com/courses/course-v1:HuaweiX+CBUCNXE088+Self-paced/courseware/f26a58a01f564dcbbc272acc319fd0f1/aaf5713e6d1442b88198ae7795eb7965/)


至此，本案例完成。